# Submit the Reference H2O Scoring Pipeline

Load the static CMK-compatible scoring pipeline, bind the reference model and golden input, submit it to the configured cluster, and inspect scored and monitoring outputs.

**Source:** Adapted from this repository's `notebooks/h2o_mojo/05_build_and_schedule_scoring_pipeline.ipynb`.

In [ ]:
from pathlib import Path
import os

from azure.ai.ml import Input, MLClient, load_job
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import ManagedIdentityConfiguration
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
bundle_value = Path(os.environ["H2O_BUNDLE_DIR"])
BUNDLE_DIR = bundle_value if bundle_value.is_absolute() else WORKSHOP_ROOT / bundle_value
MODEL_NAME = os.environ["H2O_MODEL_NAME"]
MODEL_VERSION = os.environ["H2O_MODEL_VERSION"]
ENVIRONMENT_NAME = os.environ["H2O_ENVIRONMENT_NAME"]
ENVIRONMENT_VERSION = os.environ["H2O_ENVIRONMENT_VERSION"]
COMPUTE_NAME = os.environ["AZUREML_COMPUTE_NAME"]
COMPUTE_IDENTITY_CLIENT_ID = os.environ["AZUREML_COMPUTE_IDENTITY_CLIENT_ID"].strip()
if not COMPUTE_IDENTITY_CLIENT_ID:
    raise ValueError("AZUREML_COMPUTE_IDENTITY_CLIENT_ID must identify the compute cluster UMI")
OUTPUT_DATASTORE = os.getenv("AZUREML_OUTPUT_DATASTORE", "workspaceblobstore")
RUN = os.getenv("RUN_H2O_SCORING_PIPELINE", "false").lower() in {"1", "true", "yes"}

job = load_job(WORKSHOP_ROOT / os.getenv("H2O_PIPELINE_FILE", "pipelines/h2o-customer-scoring-pipeline.yaml"))
job.inputs["model_dir"] = f"azureml:{MODEL_NAME}:{MODEL_VERSION}"
job.inputs["input_data"] = Input(type=AssetTypes.URI_FILE, path=str(BUNDLE_DIR / "golden_input.csv"))
job.inputs["correlation_id"] = "reference-workshop-run"
job.inputs["id_column"] = "__generated__"
job.inputs["fail_on_rejects"] = False
job.inputs["h2o_nthreads"] = int(os.environ["H2O_NTHREADS"])
job.inputs["h2o_max_mem_size"] = os.environ["H2O_MAX_MEM_SIZE"]
job.jobs["score"].component.environment = f"azureml:{ENVIRONMENT_NAME}:{ENVIRONMENT_VERSION}"
job.settings.default_compute = f"azureml:{COMPUTE_NAME}"
job.settings.default_datastore = f"azureml:{OUTPUT_DATASTORE}"
job.identity = ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID)
for child_job in job.jobs.values():
    child_job.identity = ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID)
job.display_name = "Reference H2O binary-model scoring"
job.tags = {"workshop": "azureml-h2o", "model": f"{MODEL_NAME}:{MODEL_VERSION}"}

assert job.identity.client_id == COMPUTE_IDENTITY_CLIENT_ID
assert all(
    child_job.identity.client_id == COMPUTE_IDENTITY_CLIENT_ID
    for child_job in job.jobs.values()
)
print(f"Runtime identity: compute cluster UMI {COMPUTE_IDENTITY_CLIENT_ID}")

if RUN:
    submitted_job = ml_client.jobs.create_or_update(job)
    print(f"Submitted: {submitted_job.name}")
    ml_client.jobs.stream(submitted_job.name)
    final_job = ml_client.jobs.get(submitted_job.name)
    if final_job.status != "Completed":
        raise RuntimeError(f"Pipeline ended with status {final_job.status}")
    print({name: output.path for name, output in final_job.outputs.items()})
else:
    print(f"Prepared scoring pipeline for {MODEL_NAME}:{MODEL_VERSION} on {COMPUTE_NAME}")
    print("Submission disabled. Set RUN_H2O_SCORING_PIPELINE=true in workshop/.env.")

## Expected Result

The CMK-compatible command pipeline completes on the configured cluster and publishes separate scored and monitoring output URIs.

Next: `../04_h2o_customer/01_package_and_validate_model.ipynb`.